# 1. Data Preparation (UNSW-NB15)

**Dataset:** UNSW-NB15 (training + testing CSVs).  
**Goal:** Load, clean, and prepare data for Standard ML and Graph ML notebooks.

**Outputs:** `prepared_data/ml_prepared.csv`, `prepared_data/tl_prepared.csv`, and `prepared_data/metadata.json`.

## Configuration

In [1]:
from pathlib import Path
import json

import numpy as np
import pandas as pd
from pydantic import BaseModel, Field
from sklearn.preprocessing import LabelEncoder


class DataPathsConfig(BaseModel):
    dataset_root: str = Field(default="UNSW-NB15 Dataset/CSV Files")
    train_file: str = Field(default="Training and Testing Sets/UNSW_NB15_training-set.csv")
    test_file: str = Field(default="Training and Testing Sets/UNSW_NB15_testing-set.csv")
    output_dir: str = Field(default="prepared_data")


class DataPrepConfig(BaseModel):
    drop_duplicates: bool = Field(default=True)
    drop_constant_columns: bool = Field(default=True)
    drop_na_rows: bool = Field(default=True)
    save_prepared_csvs: bool = Field(default=True)


class CorrelationFeatureSelectionConfig(BaseModel):
    enabled: bool = Field(default=True)
    correlation_threshold: float = Field(default=0.05)
    method: str = Field(default="spearman")


paths_cfg = DataPathsConfig()
prep_cfg = DataPrepConfig()
corr_cfg = CorrelationFeatureSelectionConfig()

print("Configuration loaded.")
print(f"dataset_root={paths_cfg.dataset_root}")
print(f"train_file={paths_cfg.train_file}")
print(f"test_file={paths_cfg.test_file}")
print(f"output_dir={paths_cfg.output_dir}")

Configuration loaded.
dataset_root=UNSW-NB15 Dataset/CSV Files
train_file=Training and Testing Sets/UNSW_NB15_training-set.csv
test_file=Training and Testing Sets/UNSW_NB15_testing-set.csv
output_dir=prepared_data


In [2]:
def resolve_existing_path(path_str: str) -> Path:
    p = Path(path_str)
    if p.exists():
        return p

    candidates = [Path.cwd() / path_str, Path.cwd().parent / path_str]
    for c in candidates:
        if c.exists():
            return c

    raise FileNotFoundError(f"Path not found: {path_str}")


dataset_root = resolve_existing_path(paths_cfg.dataset_root)
train_path = dataset_root / paths_cfg.train_file
test_path = dataset_root / paths_cfg.test_file
output_dir = resolve_existing_path(".") / paths_cfg.output_dir
output_dir.mkdir(parents=True, exist_ok=True)

train_df = pd.read_csv(train_path, low_memory=False)
test_df = pd.read_csv(test_path, low_memory=False)

train_df["split"] = "train"
test_df["split"] = "test"

unsw_df = pd.concat([train_df, test_df], ignore_index=True)
unsw_df.columns = unsw_df.columns.str.strip()

print(f"Train shape: {train_df.shape}")
print(f"Test shape: {test_df.shape}")
print(f"Merged shape: {unsw_df.shape}")
unsw_df.head(3)

Train shape: (175341, 46)
Test shape: (82332, 46)
Merged shape: (257673, 46)


,id,dur,proto,service,state,spkts,dpkts,sbytes,dbytes,rate,...,ct_dst_src_ltm,is_ftp_login,ct_ftp_cmd,ct_flw_http_mthd,ct_src_ltm,ct_srv_dst,is_sm_ips_ports,attack_cat,label,split
0,1,0.121478,tcp,-,FIN,6,4,258,172,74.087490,...,1,0,0,0,1,1,0,Normal,0,train
1,2,0.649902,tcp,-,FIN,14,38,734,42014,78.473372,...,2,0,0,0,1,6,0,Normal,0,train
2,3,1.623129,tcp,-,FIN,8,16,364,13186,14.170161,...,3,0,0,0,2,6,0,Normal,0,train


In [3]:
df = unsw_df.copy()

if "attack_cat" in df.columns:
    df["Label"] = df["attack_cat"].fillna("Normal").astype(str).str.strip()
else:
    df["Label"] = df["label"].map({0: "Normal", 1: "Attack"})

df["Label"] = df["Label"].replace({"Normal": "BENIGN"})

for c in df.columns:
    if df[c].dtype == object and c not in {"Label", "attack_cat", "proto", "service", "state", "srcip", "dstip", "split"}:
        df[c] = pd.to_numeric(df[c], errors="ignore")

numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
if numeric_cols:
    df[numeric_cols] = df[numeric_cols].replace([np.inf, -np.inf], np.nan)

rows_before = len(df)
if prep_cfg.drop_na_rows:
    df = df.dropna()
if prep_cfg.drop_duplicates:
    df = df.drop_duplicates()

print(f"Rows after cleaning: {len(df):,} (dropped {rows_before - len(df):,})")
print(df["Label"].value_counts().to_string())

Rows after cleaning: 257,673 (dropped 0)
Label
BENIGN            93000
Generic           58871
Exploits          44525
Fuzzers           24246
DoS               16353
Reconnaissance    13987
Analysis           2677
Backdoor           2329
Shellcode          1511
Worms               174


In [4]:
le = LabelEncoder()
df["label_encoded"] = le.fit_transform(df["Label"])
label_mapping = {k: int(v) for k, v in zip(le.classes_.tolist(), le.transform(le.classes_).tolist())}

graph_cols = [c for c in ["srcip", "sport", "dstip", "dsport", "proto", "state", "service", "dur", "Label", "label_encoded"] if c in df.columns]
drop_for_ml = [c for c in ["srcip", "dstip", "attack_cat", "split"] if c in df.columns]

ml_df = df.drop(columns=drop_for_ml).copy()
tl_df = df[graph_cols].copy()

if prep_cfg.drop_constant_columns:
    ml_numeric_cols = ml_df.select_dtypes(include=[np.number]).columns.tolist()
    const_cols = [c for c in ml_numeric_cols if ml_df[c].nunique(dropna=False) <= 1 and c != "label_encoded"]
    if const_cols:
        ml_df = ml_df.drop(columns=const_cols)
        print(f"Dropped {len(const_cols)} constant numeric columns from ml_df")

non_feature_cols = ["Label", "label_encoded", "attack_cat", "split", "id"]
feature_cols = [c for c in ml_df.columns if c not in non_feature_cols]

print(f"ml_df shape: {ml_df.shape}")
print(f"tl_df shape: {tl_df.shape}")
print(f"Feature columns: {len(feature_cols)}")
print(label_mapping)

ml_df shape: (257673, 46)
tl_df shape: (257673, 6)
Feature columns: 43
{'Analysis': 0, 'BENIGN': 1, 'Backdoor': 2, 'DoS': 3, 'Exploits': 4, 'Fuzzers': 5, 'Generic': 6, 'Reconnaissance': 7, 'Shellcode': 8, 'Worms': 9}


In [5]:
selected_feature_cols = None
if corr_cfg.enabled:
    numeric_feature_cols = [c for c in feature_cols if pd.api.types.is_numeric_dtype(ml_df[c]) and c != "label"]
    if numeric_feature_cols:
        corr_with_target = ml_df[numeric_feature_cols + ["label_encoded"]].corr(method=corr_cfg.method)["label_encoded"].drop("label_encoded")
        corr_abs = corr_with_target.abs().sort_values(ascending=False)
        selected_feature_cols = corr_abs[corr_abs >= corr_cfg.correlation_threshold].index.tolist()
        print(f"Selected {len(selected_feature_cols)} features with |corr| >= {corr_cfg.correlation_threshold}")
        print(selected_feature_cols[:30])

Selected 37 features with |corr| >= 0.05
['sttl', 'dload', 'ct_state_ttl', 'dmean', 'ct_dst_sport_ltm', 'dbytes', 'dpkts', 'sbytes', 'ct_src_dport_ltm', 'dloss', 'sloss', 'spkts', 'swin', 'dwin', 'rate', 'stcpb', 'dtcpb', 'dinpkt', 'dttl', 'dur', 'sinpkt', 'sload', 'ct_dst_src_ltm', 'djit', 'sjit', 'ct_srv_src', 'synack', 'tcprtt', 'ackdat', 'ct_src_ltm']


In [6]:
if prep_cfg.save_prepared_csvs:
    ml_path = output_dir / "ml_prepared.csv"
    tl_path = output_dir / "tl_prepared.csv"
    meta_path = output_dir / "metadata.json"

    ml_df.to_csv(ml_path, index=False)
    tl_df.to_csv(tl_path, index=False)

    metadata = {
        "dataset": "UNSW-NB15",
        "train_path": str(train_path),
        "test_path": str(test_path),
        "ml_shape": list(ml_df.shape),
        "tl_shape": list(tl_df.shape),
        "feature_cols": feature_cols,
        "label_mapping": label_mapping
    }
    if selected_feature_cols is not None:
        metadata["selected_feature_cols"] = selected_feature_cols
        metadata["correlation_threshold"] = corr_cfg.correlation_threshold
        metadata["correlation_method"] = corr_cfg.method

    with open(meta_path, "w", encoding="utf-8") as f:
        json.dump(metadata, f, indent=2)

    print(f"Saved ml_df: {ml_path}")
    print(f"Saved tl_df: {tl_path}")
    print(f"Saved metadata: {meta_path}")

Saved ml_df: prepared_data/ml_prepared.csv
Saved tl_df: prepared_data/tl_prepared.csv
Saved metadata: prepared_data/metadata.json
